In [1]:
# --- Cell 1: Setup ---
import pandas as pd, numpy as np
from pathlib import Path

# Works whether Jupyter starts in notebooks/ or in the project root
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "data-analytics"
RAW, PROCESSED, DB = BASE / "data" / "raw", BASE / "data" / "processed", BASE / "sql" / "agriculture.db"
print("Project folder:", BASE)

Project folder: C:\Users\User\Music\Playlists\Smart-Agriculture-Crop-Yield-Market-Intelligence-System\data-analytics


In [2]:
# --- Cell 2: Load raw data and inspect ---
raw = pd.read_csv(next(RAW.glob("Crop_Production_Statistics*.csv")))
print(raw.shape)
display(raw.head())
print(raw.dtypes)
print(raw.isnull().sum())

(345336, 8)


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75


State             str
District          str
Crop              str
Crop_Year       int64
Season            str
Area          float64
Production    float64
Yield         float64
dtype: object
State            0
District         0
Crop             9
Crop_Year        0
Season           0
Area             0
Production    4948
Yield            0
dtype: int64


In [3]:
# --- Cell 3: Strip spaces from column names and text values ---
df = raw.copy()
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={"crop_year": "year"})
for c in ["state", "district", "crop", "season"]:
    df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
print(df.columns.tolist())
print(df["season"].value_counts())

['state', 'district', 'crop', 'year', 'season', 'area', 'production', 'yield']
season
Kharif        138369
Rabi          100951
Whole Year     68680
Summer         22098
Winter          8249
Autumn          6989
Name: count, dtype: Int64


In [4]:
# --- Cell 4: Standardise state names ---
print("Before:", df["state"].nunique(), "states")
df["state"] = df["state"].replace({
    "The Dadra And Nagar Haveli": "Dadra And Nagar Haveli",
    "Andaman And Nicobar Island": "Andaman And Nicobar Islands",
    "Laddak": "Ladakh",
})
print("After :", df["state"].nunique(), "states")
print(sorted(df["state"].unique()))

Before: 37 states
After : 36 states


['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli', 'Daman And Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [5]:
# --- Cell 5: Missing and zero values ---
# Missing Production is NOT a crop failure. Do not fill it with Area x Yield
# (Yield is 0 on exactly those rows, so you would create fake zeros).
miss = df[df["production"].isna()]
print("Rows with missing production:", len(miss), "| of them with Yield == 0:", (miss["yield"] == 0).sum())
print("Rows with zero production   :", (df["production"] == 0).sum())

n = len(df)
df = df.dropna(subset=["crop", "area", "production"])
df = df[(df["area"] > 0) & (df["production"] > 0)]
print(f"Dropped {n - len(df):,} rows, remaining {len(df):,}")

Rows with missing production: 4948 | of them with Yield == 0: 4948
Rows with zero production   : 1466


Dropped 6,418 rows, remaining 338,918


In [6]:
# --- Cell 6: Recompute Yield and remove duplicates ---
df["yield"] = df["production"] / df["area"]
key = ["state", "district", "crop", "year", "season"]
print("Duplicates:", df.duplicated(key).sum())
df = df.drop_duplicates(key)
print("Rows now:", len(df))

Duplicates: 584


Rows now: 338334


In [7]:
# --- Cell 7: Outliers (per crop, because units differ) ---
print(df.groupby("crop")["yield"].median().sort_values(ascending=False).head(6))
print("\nMax yield before:", df["yield"].max().round(1))

q1 = df.groupby("crop")["yield"].transform(lambda s: s.quantile(0.25))
q3 = df.groupby("crop")["yield"].transform(lambda s: s.quantile(0.75))
n = len(df)
df = df[df["yield"] <= q3 + 3 * (q3 - q1)]
print(f"Removed {n - len(df):,} outliers | max yield now: {df['yield'].max():.1f}")

crop
Coconut      8079.016929
Sugarcane      55.454495
Banana         23.529864
Tapioca        12.900000
Onion          11.000000
Potato         10.772586
Name: yield, dtype: float64

Max yield before: 43958.3


Removed 3,704 outliers | max yield now: 30093.3


In [8]:
# --- Cell 8: Final check and save ---
df["yield_unit_note"] = df["crop"].map(lambda c: "nuts/ha" if c == "Coconut" else "tonnes/ha")
df = df[["state", "district", "crop", "year", "season", "area", "production", "yield", "yield_unit_note"]]

assert df.isna().sum().sum() == 0 and (df["area"] > 0).all() and not df.duplicated(key).any()
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "cleaned_crop_data.csv", index=False)
print(f"Saved {len(df):,} rows ({len(df)/len(raw):.1%} of raw)")
df.describe().round(2)

Saved 334,630 rows (96.9% of raw)


,year,area,production,yield
count,334630.00,334630.00,3.346300e+05,334630.00
mean,2008.87,11981.80,9.699751e+05,80.27
std,6.57,46490.24,2.170321e+07,913.79
min,1997.00,0.01,1.000000e+00,0.00
25%,2003.00,82.00,8.900000e+01,0.57
50%,2009.00,580.00,7.210000e+02,1.01
75%,2015.00,4348.00,7.200000e+03,2.42
max,2020.00,8580100.00,1.597800e+09,30093.33
